# 11 — Proposal Writer
> Paste in any RFP and get back a fully drafted proposal — executive summary, methodology, team credentials, timeline, commercial terms, differentiators, and a compliance statement — in seconds.

*Run all cells. Swap the input in the final code cell with your own data.*

In [ ]:
%pip install -q langchain-openai langchain-core pydantic python-dotenv
import os
os.environ['OPENAI_API_KEY'] = 'sk-...'  # replace

In [ ]:
from typing import List, Optional
from pydantic import BaseModel, Field
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI


# --- Data shapes ---

class RFPRequirement(BaseModel):
    section: str = Field(description="Which section or question this requirement comes from")
    requirement: str = Field(description="The specific thing the client is asking for")
    mandatory: bool = Field(description="True if this is a pass/fail requirement")


class ProposalOutline(BaseModel):
    client_name: Optional[str] = None
    rfp_title: str
    submission_deadline: Optional[str] = None
    requirements: List[RFPRequirement] = Field(
        description="All requirements extracted from the RFP, mandatory ones first"
    )
    win_themes: List[str] = Field(
        description="2-4 strategic themes that should run through the whole proposal"
    )
    evaluation_criteria: List[str] = Field(
        description="How the client will score proposals, in priority order"
    )
    sections_to_write: List[str] = Field(
        description="Ordered list of proposal sections that need to be drafted"
    )


class Proposal(BaseModel):
    client_name: Optional[str] = None
    rfp_title: str
    win_themes: List[str]
    executive_summary: str = Field(
        description="Why we win -- lead with the client's problem and our unique answer"
    )
    our_approach: str = Field(
        description="Methodology and how we would deliver the work"
    )
    team_and_credentials: str = Field(
        description="Relevant experience, key personnel, and why we are best placed to deliver"
    )
    timeline: str = Field(
        description="Phased project plan with key milestones and deliverables"
    )
    commercial: str = Field(
        description="Pricing structure, value drivers, and commercial terms"
    )
    why_us: str = Field(
        description="Differentiators -- what we offer that competitors cannot match"
    )
    key_differentiators: List[str] = Field(
        description="3-5 bullet-point differentiators for the cover page or summary slide"
    )
    compliance_statement: str = Field(
        description="Confirmation that all mandatory requirements are met"
    )


# --- Agent prompts ---

SUPERVISOR_SYSTEM = SystemMessage(
    """You are a proposal director reviewing an RFP before your team drafts a response.

Your job is to produce a structured decomposition of the RFP:
1. Extract every requirement, marking which are mandatory pass/fail criteria
2. Identify 2-4 win themes -- the strategic angles that should run through the whole proposal
3. Identify how the client will evaluate and score proposals
4. List the sections the proposal should contain, in order

Be precise. Requirements you miss now become compliance failures later."""
)

WRITER_SYSTEM = SystemMessage(
    """You are a senior proposal writer crafting a winning RFP response.

You will receive:
- The original RFP text
- A structured outline (win themes, requirements, evaluation criteria)

Your draft must:
- Lead every section with the client's problem, not your firm's capabilities
- Weave the win themes consistently through all sections
- Be specific about methodology -- no generic consulting language
- Address all mandatory requirements explicitly in the compliance_statement
- Keep the executive_summary under 200 words and punchy

Write to win, not to comply."""
)


# --- Pipeline ---

def _outline(llm, rfp_text: str) -> ProposalOutline:
    supervisor = SUPERVISOR_SYSTEM | llm.with_structured_output(ProposalOutline)
    return supervisor.invoke(HumanMessage(content="RFP to decompose:\n\n" + rfp_text))


def _draft(llm, rfp_text: str, outline: ProposalOutline) -> Proposal:
    context = (
        "RFP text:\n" + rfp_text + "\n\n"
        "Win themes: " + ", ".join(outline.win_themes) + "\n"
        "Evaluation criteria: " + ", ".join(outline.evaluation_criteria) + "\n"
        "Mandatory requirements:\n"
        + "\n".join(
            "- [" + r.section + "] " + r.requirement
            for r in outline.requirements
            if r.mandatory
        )
    )
    writer = WRITER_SYSTEM | llm.with_structured_output(Proposal)
    return writer.invoke(context)


def run(rfp_text: str) -> dict:
    """Two-agent proposal writer. Returns outline and full proposal draft."""
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    outline = _outline(llm, rfp_text)
    proposal = _draft(llm, rfp_text, outline)
    return {"outline": outline, "proposal": proposal}


print("Ready.")

## The scenario

A regional NHS trust has issued an RFP for a workforce analytics platform. They need a vendor to assess their current HR data infrastructure, recommend a technology solution, and deliver a working pilot within 12 weeks. The RFP has hard pass/fail requirements around ISO 27001 certification, UK data residency, and NHS Digital accreditation. We'll run the full pipeline and see what the agent returns.

In [ ]:
SAMPLE_RFP = """
REQUEST FOR PROPOSAL
Workforce Analytics Platform — Discovery, Recommendation and Pilot
Issued by: Northgate NHS Foundation Trust
RFP Reference: NHT-2025-WFA-004
Submission Deadline: 14 April 2025

1. BACKGROUND
Northgate NHS Foundation Trust employs 6,200 staff across acute, community, and
mental health services. Current workforce data is held across three legacy HR systems
with no unified reporting layer. The People Directorate cannot produce timely
absence, turnover, or bank-versus-substantive staffing reports without manual
extraction from each system. Executive leadership has ring-fenced GBP 220,000 to
procure a modern workforce analytics solution and move to data-driven rostering
decisions by Q3 2025.

2. SCOPE OF WORK
The selected supplier must deliver:
a) Current-State Assessment: Audit all three legacy HR systems, data quality,
   and existing reporting workflows across the People Directorate.
b) Platform Recommendation: Evaluate at least three workforce analytics platforms
   against NHS Digital standards and Trust-specific requirements; present a
   scored recommendation with rationale.
c) Integration Design: Produce a data architecture design connecting the chosen
   platform to the Trust's ESR (Electronic Staff Record) and bank-staff system.
d) Pilot Delivery: Stand up a working pilot covering absence and turnover
   dashboards for two directorates within 12 weeks of contract signature.
e) Knowledge Transfer: Deliver hands-on training to a minimum of 10 People
   Directorate staff and produce runbook documentation.

3. MANDATORY REQUIREMENTS (PASS/FAIL)
M1. Supplier must hold current ISO 27001 certification.
M2. All data must remain within UK data centres; no cross-border transfer permitted.
M3. At least one named team member must hold NHS Digital Data Security and
    Protection Toolkit accreditation.
M4. Supplier must evidence at least two prior workforce analytics implementations
    for NHS or equivalent regulated public-sector health bodies.
M5. Proposal must be fixed-price; day-rate or time-and-materials bids will be
    rejected.

4. EVALUATION CRITERIA (scored out of 100)
- Prior NHS / regulated health-sector experience and case studies: 40 points
- Technical approach and integration methodology: 25 points
- Team credentials and named personnel: 20 points
- Value for money and fixed-price transparency: 15 points

5. COMMERCIAL
Total budget is GBP 220,000 inclusive of software licences for the pilot period.
Northgate reserves the right to extend the engagement for full rollout subject
to a separately negotiated contract.

6. SUBMISSION REQUIREMENTS
- Maximum 15 pages excluding appendices
- Submit to procurement@northgatenhsft.nhs.uk by 14 April 2025 17:00 BST
"""

result = run(SAMPLE_RFP)
outline = result["outline"]
proposal = result["proposal"]

# --- Stage 1: what the supervisor extracted ---
print("=" * 60)
print("STAGE 1 — RFP BREAKDOWN")
print("=" * 60)
print(f"Title:    {outline.rfp_title}")
print(f"Client:   {outline.client_name or 'Unknown'}")
print(f"Deadline: {outline.submission_deadline or 'Not specified'}")

mandatory = [r for r in outline.requirements if r.mandatory]
optional  = [r for r in outline.requirements if not r.mandatory]
print(f"\nRequirements: {len(mandatory)} mandatory, {len(optional)} optional")

print("\nMandatory pass/fail:")
for r in mandatory:
    print(f"  [{r.section}] {r.requirement}")

print(f"\nWin themes ({len(outline.win_themes)}):")
for t in outline.win_themes:
    print(f"  * {t}")

print("\nEvaluation criteria (ranked):")
for i, c in enumerate(outline.evaluation_criteria, 1):
    print(f"  {i}. {c}")

# --- Stage 2: the drafted proposal ---
print()
print("=" * 60)
print("STAGE 2 — PROPOSAL DRAFT")
print("=" * 60)

print("\n--- EXECUTIVE SUMMARY ---")
print(proposal.executive_summary)

print("\n--- OUR APPROACH ---")
print(proposal.our_approach)

print("\n--- TIMELINE ---")
print(proposal.timeline)

print("\n--- COMMERCIAL ---")
print(proposal.commercial)

print("\n--- KEY DIFFERENTIATORS ---")
for d in proposal.key_differentiators:
    print(f"  * {d}")

print("\n--- COMPLIANCE STATEMENT ---")
print(proposal.compliance_statement)

## Use your own data

Replace `SAMPLE_RFP` above with:
- Your RFP text (paste directly as a Python string)
- Or read from a file: `SAMPLE_RFP = open('your_rfp.txt').read()`

The agent will return a complete proposal draft structured around your RFP's own mandatory requirements, evaluation criteria, and win themes.